## Steps

1. Loading data  
2. EDA  
3. Data Preprocessing  
        3.1 Cleaning  
        3.2 Stemming  
        3.3 All together  
        3.4 Target Encoding  
4. Tokens Visualization  
5. Vectorization  
        5.1 Tuning CountVectorizer  
        5.2 TF‑IDF  
        5.3 Word Embedding: GloVe  
6. Modelling  
        6.1 Naive Bayes (Document-Term Matrix)  
        6.2 Naive Bayes (TF‑IDF)  
        6.3 XGBoost  
7. LSTM  
8. BERT  
9. NLP: Disaster tweets  
        9.1 EDA


In [25]:
pip install nbformat>=4.2.0

Note: you may need to restart the kernel to use updated packages.


In [26]:
import re
import string
import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from plotly import graph_objs as go
import plotly.express as px
import plotly.figure_factory as ff
from collections import Counter
import nltk
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from PIL import Image
nltk.download('stopwords') ## Downloading Stopword corpus
nltk.download('punkt') ## Downloading Tokenization
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import os
import spacy
import random
from spacy.util import compounding
from spacy.util import minibatch
from collections import defaultdict
from collections import Counter
import keras
from keras.models import Sequential
from keras.initializers import Constant
from keras.layers import (LSTM,
                          Embedding,
                          BatchNormalization,
                          Dense,
                          TimeDistributed,
                          Dropout,
                          Bidirectional,
                          Flatten,
                          GlobalMaxPool1D)
# from keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Embedding
# from keras.layers.embeddings import Embedding
from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    accuracy_score
)

[nltk_data] Downloading package stopwords to C:\Users\Satish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Satish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [27]:
primary_blue="#496595"
primary_blue2="#85a1c1"
primary_blue3="#3f4d63"
primary_grey="#c6ccd8"
primary_black="#202022"
primary_bgcolor="f4f0ea"

primary_green=px.colors.qualitative.Plotly[2]

In [28]:
df = pd.read_csv("D:\SummerPEP\pep_genai\data\spam.csv", encoding='latin-1')

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_13320\84810637.py:1: SyntaxWarning: invalid escape sequence '\S'
  df = pd.read_csv("D:\SummerPEP\pep_genai\data\spam.csv", encoding='latin-1')


In [29]:
df.head

<bound method NDFrame.head of         v1                                                 v2 Unnamed: 2  \
0      ham  Go until jurong point, crazy.. Available only ...        NaN   
1      ham                      Ok lar... Joking wif u oni...        NaN   
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3      ham  U dun say so early hor... U c already then say...        NaN   
4      ham  Nah I don't think he goes to usf, he lives aro...        NaN   
...    ...                                                ...        ...   
5567  spam  This is the 2nd time we have tried 2 contact u...        NaN   
5568   ham              Will Ì_ b going to esplanade fr home?        NaN   
5569   ham  Pity, * was in mood for that. So...any other s...        NaN   
5570   ham  The guy did some bitching but I acted like i'd...        NaN   
5571   ham                         Rofl. Its true to its name        NaN   

     Unnamed: 3 Unnamed: 4  
0           NaN        NaN  

In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 677.1 KB


In [31]:
df.describe()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
count,5572,5572,50,12,6
unique,2,5169,43,10,5
top,ham,"Sorry, I'll call later","bt not his girlfrnd... G o o d n i g h t . . .@""","MK17 92H. 450Ppw 16""","GNT:-)"""
freq,4825,30,3,2,2


In [32]:
len(df.columns)

5

In [33]:
df = df.dropna(how='any', axis=1)
df.columns=['target','message']
df


,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [34]:
df.head()

,target,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [35]:
df['message_len'] = df['message'].apply(lambda x: len(x.split(' ')))
df.head()

,target,message,message_len
0,ham,"Go until jurong point, crazy.. Available only ...",20
1,ham,Ok lar... Joking wif u oni...,6
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28
3,ham,U dun say so early hor... U c already then say...,11
4,ham,"Nah I don't think he goes to usf, he lives aro...",13


In [36]:
print("Max Length of message: ", max(df['message_len']))
print("Min Length of message: ", min(df['message_len']))

Max Length of message:  171
Min Length of message:  1


In [37]:
df['message'].apply(lambda x: len(x.split('. ')))

0       3
1       2
2       2
3       2
4       1
       ..
5567    4
5568    1
5569    2
5570    1
5571    2
Name: message, Length: 5572, dtype: int64

In [38]:
df['target'].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

In [39]:
# balance_counts=pd.DataFrame({'target': df['target'].value_counts().index, 'count': df['target'].value_counts().values})

In [40]:
# balance_counts

In [41]:
# print(balance_counts['count'][0])

In [42]:
balance_counts = df['target'].value_counts()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['ham'],
    y=[balance_counts.get('ham', 0)],
    name='ham',
    text=[balance_counts.get('ham', 0)],
    textposition='auto',
    marker_color=primary_blue
))
fig.add_trace(go.Bar(
    x=['spam'],
    y=[balance_counts.get('spam', 0)],
    name='spam',
    text=[balance_counts.get('spam', 0)],
    textposition='auto',
    marker_color=primary_grey
))
fig.update_layout(
    title='<span style="font-size:32px; font-family:Times New Roman">Dataset distribution by target</span>'
)
fig.show()

In [43]:
ham_df = df[df['target'] == 'ham']['message_len'].value_counts().sort_index()
spam_df = df[df['target'] == 'spam']['message_len'].value_counts().sort_index()
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ham_df.index,
    y=ham_df.values,
    name='ham',
    fill='tozeroy',
    marker_color=primary_blue
))
fig.add_trace(go.Scatter(
    x=spam_df.index,
    y=spam_df.values,
    name='spam',
    fill='tozeroy',
    marker_color=primary_grey
))
fig.update_layout(
    title='<span style="font-size:32px; font-family:Times New Roman">Dataset distribution by target</span>'
)
fig.update_xaxes(range=[0,70])
fig.show()

In [44]:
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

df['message_clean'] = df ['message'].apply(clean_text)
df.head()

<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:10: SyntaxWarning: invalid escape sequence '\w'
<>:5: SyntaxWarning: invalid escape sequence '\['
<>:6: SyntaxWarning: invalid escape sequence '\S'
<>:10: SyntaxWarning: invalid escape sequence '\w'
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_13320\2380423804.py:5: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_13320\2380423804.py:6: SyntaxWarning: invalid escape sequence '\S'
  text = re.sub('https?://\S+|www\.\S+', '', text)
C:\Users\Satish Kumar\AppData\Local\Temp\ipykernel_13320\2380423804.py:10: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


,target,message,message_len,message_clean
0,ham,"Go until jurong point, crazy.. Available only ...",20,go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,6,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,28,free entry in a wkly comp to win fa cup final...
3,ham,U dun say so early hor... U c already then say...,11,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",13,nah i dont think he goes to usf he lives aroun...


In [49]:
stop_words = stopwords.words('english')
more_stopwords = ['u', 'im', 'c']
stop_words = stop_words + more_stopwords

def remove_stopwords(text):
    text = ' '.join(word for word in text.split(' ') if word not in stop_words)
    return text

df['message'] = df['message'].apply(remove_stopwords)
df.head()

,target,message,message_len,message_clean
0,ham,"Go jurong point, crazy.. Available bugis n gre...",20,go until jurong point crazi avail onli in bugi...
1,ham,Ok lar... Joking wif oni...,6,ok lar joke wif u oni
2,spam,Free entry 2 wkly comp win FA Cup final tkts 2...,28,free entri in a wkli comp to win fa cup final...
3,ham,U dun say early hor... U already say...,11,u dun say so earli hor u c alreadi then say
4,ham,"Nah I think goes usf, lives around though",13,nah i dont think he goe to usf he live around ...


In [50]:
stemmer = nltk.SnowballStemmer("english")

def stemm_text(text):
    text = ' '.join(stemmer.stem(word) for word in text.split(' '))
    return text

In [51]:
df['message_clean'] = df['message_clean'].apply(stemm_text)
df.head()

,target,message,message_len,message_clean
0,ham,"Go jurong point, crazy.. Available bugis n gre...",20,go until jurong point crazi avail on in bugi n...
1,ham,Ok lar... Joking wif oni...,6,ok lar joke wif u oni
2,spam,Free entry 2 wkly comp win FA Cup final tkts 2...,28,free entri in a wkli comp to win fa cup final...
3,ham,U dun say early hor... U already say...,11,u dun say so ear hor u c alreadi then say
4,ham,"Nah I think goes usf, lives around though",13,nah i dont think he goe to usf he live around ...


In [52]:
def preprocess_data(text):
    text=clean_text(text)
    text= ' '.join(word for word in text.split(' ') if word not in stop_words)
    text= ' '.join(stemmer.stem(word) for word in text.split())
    return text

In [53]:
df['message'] = df['message'].apply(preprocess_data)
df.head()

,target,message,message_len,message_clean
0,ham,go jurong point crazi avail bugi n great world...,20,go until jurong point crazi avail on in bugi n...
1,ham,ok lar joke wif oni,6,ok lar joke wif u oni
2,spam,free entri wkli comp win fa cup final tkts may...,28,free entri in a wkli comp to win fa cup final...
3,ham,dun say earli hor alreadi say,11,u dun say so ear hor u c alreadi then say
4,ham,nah think goe usf live around though,13,nah i dont think he goe to usf he live around ...


In [54]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['target_encoded'] = le.fit_transform(df['target'])
print('Classes:', list(le.classes_))
df.head()

Classes: ['ham', 'spam']


,target,message,message_len,message_clean,target_encoded
0,ham,go jurong point crazi avail bugi n great world...,20,go until jurong point crazi avail on in bugi n...,0
1,ham,ok lar joke wif oni,6,ok lar joke wif u oni,0
2,spam,free entri wkli comp win fa cup final tkts may...,28,free entri in a wkli comp to win fa cup final...,1
3,ham,dun say earli hor alreadi say,11,u dun say so ear hor u c alreadi then say,0
4,ham,nah think goe usf live around though,13,nah i dont think he goe to usf he live around ...,0


In [55]:
x=df['message']
y=df['target']

print(len(x), len(y))

5572 5572


In [56]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y , random_state=42)
print(len(x_train),len(y_train))
print(len(x_test),len(y_test))

4179 4179
1393 1393


In [57]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()
X_train = vect.fit_transform(x_train)
X_test = vect.transform(x_test)
print('Vocabulary size:', len(vect.vocabulary_))

Vocabulary size: 5713


In [58]:
vect_tunned = CountVectorizer(ngram_range=(1,2), min_df=0.1, max_df=0.7, max_features=100)

In [59]:
from sklearn.feature_extraction.text import TfidfTransformer

x_train_dtm = X_train
x_test_dtm = X_test
tfidf_transformer = TfidfTransformer()
tfidf_transformer.fit(x_train_dtm)
x_train_tfidf = tfidf_transformer.transform(x_train_dtm)
x_test_tfidf = tfidf_transformer.transform(x_test_dtm)

print('TF-IDF shape:', x_train_tfidf.shape)
print("Test TF-IDF shape:", x_test_tfidf.shape)

TF-IDF shape: (4179, 5713)
Test TF-IDF shape: (1393, 5713)


In [60]:
#GLOVE embedding
texts=df['message']
target=df['target_encoded']


In [61]:
from tensorflow.keras.preprocessing.text import Tokenizer


word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(texts)

vocab_length = len(word_tokenizer.word_index) + 1
print(vocab_length)

6763


In [62]:
def embed(corpus):
    return word_tokenizer.texts_to_sequences(corpus)

longest_train = max(texts, key=lambda sentence: len(sentence.split()))
length_long_sentence = len(longest_train.split())

train_padded_sentences = pad_sequences(
    embed(texts),
    length_long_sentence,
    padding='post'
)


train_padded_sentences

array([[   2, 3196,  273, ...,    0,    0,    0],
       [   7,  236,  528, ...,    0,    0,    0],
       [   8,  357,  587, ...,    0,    0,    0],
       ...,
       [6761, 1001, 6762, ...,    0,    0,    0],
       [ 137, 1251, 1602, ...,    0,    0,    0],
       [1991,  378,  167, ...,    0,    0,    0]],
      shape=(5572, 79), dtype=int32)

In [63]:
embeddings_dictionary = {}
embedding_dim = 100

with open(r"D:\SummerPEP\pep_genai\datasets\glove.6B.100d.txt\glove.6B.100d.txt", encoding="utf-8", errors="ignore") as fp:
    for line in fp.readlines():
        records = line.split()
        word = records[0]
        vector_dimensions = np.asarray(records[1:], dtype="float32")
        embeddings_dictionary[word] = vector_dimensions

In [64]:
embedding_matrix= np.zeros((vocab_length, embedding_dim))
for word,index in word_tokenizer.word_index.items():
    embedding_vector = embeddings_dictionary.get(word)
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector
embedding_matrix

array([[ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ],
       [-0.57832998, -0.0036551 ,  0.34658   , ...,  0.070204  ,
         0.44509   ,  0.24147999],
       [-0.078894  ,  0.46160001,  0.57779002, ...,  0.26352   ,
         0.59397   ,  0.26741001],
       ...,
       [ 0.63009   , -0.036992  ,  0.24052   , ...,  0.10029   ,
         0.056822  ,  0.25018999],
       [-0.12002   , -1.23870003, -0.23303001, ...,  0.13658001,
        -0.61848003,  0.049843  ],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
         0.        ,  0.        ]], shape=(6763, 100))

In [65]:
import plotly.express as px
x_axes = ["ham", "spam"]
y_axes = ["spam", "ham"]

def conf_matrix(z, x=x_axes, y=y_axes):
    # Flip matrix for proper visualization
    z = np.flip(z, 0)

    # Convert values to strings for annotations
    z_text = [[str(value) for value in row] for row in z]

    fig = ff.create_annotated_heatmap(
        z=z,
        x=x,
        y=y,
        annotation_text=z_text,
        colorscale="Viridis"
    )

    fig.update_layout(
        title="<b>Confusion Matrix</b>",
        xaxis_title="Predicted Value",
        yaxis_title="Actual Value"
    )

    fig['data'][0]['showscale'] = True
    fig.show()


In [66]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(x_train_dtm,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3623., 556.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.02]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 5713)","[[1.,3.,1.,...,0.,7.,1.], [0.,0.,0.,...,1.,0.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 5713)","[[ -9.7 , -9. , -9.7 ,...,-10.39, -8.31, -9.7 ], [ -9.44, -9.44, -9.44,..., -8.75, -9.44, -9.44]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5713


In [67]:
y_pred_class = nb.predict(x_test_dtm)
y_pred_prob = nb.predict_proba(x_test_dtm)[:, 1]

In [68]:
from sklearn import metrics

# Confusion Matrix
cm = metrics.confusion_matrix(y_test, y_pred_class)
conf_matrix(cm)

In [69]:
print("Accuracy:", metrics.accuracy_score(y_test, y_pred_class))

Accuracy: 0.9798994974874372


In [70]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

pipe=Pipeline([('bow',CountVectorizer()),
               ('tfidf',TfidfTransformer()),
               ('model', MultinomialNB())])
pipe.fit(x_train,y_train)
y_pred_class=pipe.predict(x_test)
print(metrics.accuracy_score(y_test,y_pred_class))
conf_matrix(metrics.confusion_matrix(y_test,y_pred_class))

0.9597989949748744


In [71]:
import xgboost as xgb
pipe=Pipeline(
    [
    ('bow', CountVectorizer()),
    ('tfid', TfidfTransformer()),
    ('model', xgb.XGBClassifier(
        learning_rate=0.1,
        max_depth=7,
        n_estimators=80,
        use_label_encoder=False,
        eval_metric='auc',
    ))
    
])

# LSTM

In [72]:
x_train,x_test,y_train,y_test = train_test_split(
    train_padded_sentences,
    target,test_size=0.25
)

In [73]:
train_padded_sentences

array([[   2, 3196,  273, ...,    0,    0,    0],
       [   7,  236,  528, ...,    0,    0,    0],
       [   8,  357,  587, ...,    0,    0,    0],
       ...,
       [6761, 1001, 6762, ...,    0,    0,    0],
       [ 137, 1251, 1602, ...,    0,    0,    0],
       [1991,  378,  167, ...,    0,    0,    0]],
      shape=(5572, 79), dtype=int32)

In [74]:
def glove_lstm():
    model = Sequential()
    model.add(Embedding(
        input_dim=embedding_matrix.shape[0], 
        output_dim=embedding_matrix.shape[1], 
        weights = [embedding_matrix], 
        input_length=length_long_sentence
    ))
    model.add(Bidirectional(LSTM(
        length_long_sentence, 
        return_sequences = True, 
        recurrent_dropout=0.2
    )))
    model.add(GlobalMaxPool1D())
    model.add(BatchNormalization())
    model.add(Dropout(0.5))
    model.add(Dense(length_long_sentence, activation = "relu"))
    model.add(Dropout(0.5))
    model.add(Dense(length_long_sentence, activation = "relu"))
    model.add(Dropout(0.5))
    model.add(Dense(1, activation = 'sigmoid'))
    model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
    return model

model = glove_lstm()
model.summary()


d:\SummerPEP\pep_genai\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       676,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 676,300 (2.58 MB)

 Trainable params: 676,300 (2.58 MB)

 Non-trainable params: 0 (0.00 B)

In [75]:
model = glove_lstm()
checkpoint = ModelCheckpoint(
    'model.h5',
    monitor='val_loss',
    verbose=1,
    save_best_only=True
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    verbose=1,
    patience=5,
    min_lr=0.001
)
history = model.fit(
    x_train,
    y_train,
    epochs=7,
    batch_size=32,
    validation_data=(x_test,y_test),
    verbose=1,
    callbacks=(reduce_lr,checkpoint)
)

Epoch 1/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - accuracy: 0.8784 - loss: 0.3261
Epoch 1: val_loss improved from None to 0.32778, saving model to model.h5



Epoch 1: finished saving model to model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 37s 180ms/step - accuracy: 0.8784 - loss: 0.3261 - val_accuracy: 0.9663 - val_loss: 0.3278 - learning_rate: 0.0010
Epoch 2/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.9342 - loss: 0.1871
Epoch 2: val_loss improved from 0.32778 to 0.12426, saving model to model.h5



Epoch 2: finished saving model to model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 129ms/step - accuracy: 0.9342 - loss: 0.1871 - val_accuracy: 0.9785 - val_loss: 0.1243 - learning_rate: 0.0010
Epoch 3/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step - accuracy: 0.9562 - loss: 0.1399
Epoch 3: val_loss improved from 0.12426 to 0.06479, saving model to model.h5



Epoch 3: finished saving model to model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 34s 255ms/step - accuracy: 0.9562 - loss: 0.1399 - val_accuracy: 0.9813 - val_loss: 0.0648 - learning_rate: 0.0010
Epoch 4/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9684 - loss: 0.1051
Epoch 4: val_loss did not improve from 0.06479
131/131 ━━━━━━━━━━━━━━━━━━━━ 13s 101ms/step - accuracy: 0.9684 - loss: 0.1051 - val_accuracy: 0.9390 - val_loss: 0.1640 - learning_rate: 0.0010
Epoch 5/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.9720 - loss: 0.0985
Epoch 5: val_loss improved from 0.06479 to 0.05060, saving model to model.h5



Epoch 5: finished saving model to model.h5
131/131 ━━━━━━━━━━━━━━━━━━━━ 15s 115ms/step - accuracy: 0.9720 - loss: 0.0985 - val_accuracy: 0.9871 - val_loss: 0.0506 - learning_rate: 0.0010
Epoch 6/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.9763 - loss: 0.0815
Epoch 6: val_loss did not improve from 0.05060
131/131 ━━━━━━━━━━━━━━━━━━━━ 22s 167ms/step - accuracy: 0.9763 - loss: 0.0815 - val_accuracy: 0.9885 - val_loss: 0.0548 - learning_rate: 0.0010
Epoch 7/7
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step - accuracy: 0.9847 - loss: 0.0605
Epoch 7: val_loss did not improve from 0.05060
131/131 ━━━━━━━━━━━━━━━━━━━━ 30s 227ms/step - accuracy: 0.9847 - loss: 0.0605 - val_accuracy: 0.9856 - val_loss: 0.0745 - learning_rate: 0.0010


In [76]:
# Print the final epoch's accuracies
print(f"Final Training Accuracy:   {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_loss'][-1]:.4f}")

Final Training Accuracy:   0.9847
Final Validation Accuracy: 0.0745


In [ ]:
def plot_learning_curve(history,arr):
    fig,ax = plt.subplots(1,2,figsize=(20,5))
    for idx in range(2):
        ax[idx].plot(history.history[arr[idx][0]])
        ax[idx].plot(history.history[arr[idx][1]])
        ax[idx].legend